In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"


Mounted at /content/drive


In [2]:
import json
import os
import platform
import time

import numpy as np
import psutil
import torch
from google.colab import drive

drive.mount("/content/drive")

MODERNBERT_DIR = "/content/drive/MyDrive/task2_final"
TASK3_DIR = "/content/drive/MyDrive/task3_cpu_deployment"
QWEN_SAVE_DIR = "/content/drive/MyDrive/task2_qlora_qwen"
QWEN_CKPT_DIR = f"{QWEN_SAVE_DIR}/checkpoints"


EVAL_MODEL_PATH = f"{MODERNBERT_DIR}/fold0_evaluation_model"
MODERNBERT_CKPT_PATH = f"{MODERNBERT_DIR}/final_all_data_model"
BENCHMARK_DIR = f"{TASK3_DIR}/benchmarks"
os.makedirs(BENCHMARK_DIR, exist_ok=True)

torch.set_num_threads(2)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

print("Logical CPUs:", psutil.cpu_count(logical=True))
print("Physical CPUs:", psutil.cpu_count(logical=False))
print("PyTorch threads:", torch.get_num_threads())
print("CPU:", platform.processor() or "Intel Xeon @ 2.20GHz")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Logical CPUs: 2
Physical CPUs: 1
PyTorch threads: 2
CPU: x86_64


In [ ]:
!pip uninstall -y diffusers

Found existing installation: diffusers 0.40.0
Uninstalling diffusers-0.40.0:
  Successfully uninstalled diffusers-0.40.0


In [ ]:
!pip install -q transformers "accelerate<1.10" onnxruntime "optimum[onnxruntime]" huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.1/367.1 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 13.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have hugg

In [ ]:
!lscpu | grep -i "model name"
!cat /proc/cpuinfo | grep flags | head -1 | grep -o "avx512_vnni" || echo "No AVX512-VNNI support detected"

Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
No AVX512-VNNI support detected


In [ ]:
%pip uninstall -y optimum optimum-onnx
%pip install --no-cache-dir --upgrade \
    "optimum-onnx[onnxruntime]" \
    transformers \
    "accelerate<1.10" \
    huggingface_hub

Found existing installation: optimum 2.1.0
Uninstalling optimum-2.1.0:
  Successfully uninstalled optimum-2.1.0
Found existing installation: optimum-onnx 0.1.0
Uninstalling optimum-onnx-0.1.0:
  Successfully uninstalled optimum-onnx-0.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 152.4 MB/s eta 0:00:00


In [ ]:
from optimum.onnxruntime import ORTModelForSequenceClassification
print("✅ ORTModelForSequenceClassification imports cleanly")

import torch, transformers, onnxruntime, accelerate
from importlib.metadata import version
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"accelerate: {accelerate.__version__}")
print(f"onnxruntime: {onnxruntime.__version__}")
print(f"optimum: {version('optimum')}")
print(f"CUDA available: {torch.cuda.is_available()}")

✅ ORTModelForSequenceClassification imports cleanly
torch: 2.11.0+cpu
transformers: 4.57.6
accelerate: 1.9.0
onnxruntime: 1.29.0
optimum: 2.1.0
CUDA available: False


## Initial all-data ModernBERT load

This loads `final_all_data_model`, which is the production-serving checkpoint trained on all 1,000 examples. It is appropriate for demonstration inference but not for unbiased accuracy measurement.

In [3]:
for d in [f"{TASK3_DIR}/gguf", f"{TASK3_DIR}/modernbert_onnx",
          f"{TASK3_DIR}/qwen_classifier_cpu", f"{TASK3_DIR}/benchmarks"]:
    os.makedirs(d, exist_ok=True)

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import time

mb_tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
mb_model = AutoModelForSequenceClassification.from_pretrained(MODERNBERT_CKPT_PATH)
mb_model.eval()
print(f"ModernBERT reloaded, id2label entries: {len(mb_model.config.id2label)}")

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

ModernBERT reloaded, id2label entries: 99


##Genuine fold-0 held-out evaluation

The exact label encoding and stratified split from Task 2 are reconstructed here. `fold0_evaluation_model` was trained on the 800 training indices, so the remaining 200 examples provide an honest deployment evaluation set.

In [ ]:
from datasets import load_dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import numpy as np

ds = load_dataset("FareedKhan/1k_stories_100_genre")["train"]
df = ds.to_pandas()

# Reproduce the EXACT same label encoding and fold split used during training
label_encoder = LabelEncoder()
df['genre_label'] = label_encoder.fit_transform(df['genre'])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(df['story'], df['genre_label']))

train_idx, val_idx = folds[0]  # fold 0 — same split used throughout your Task 2 work
eval_sample = df.iloc[val_idx].reset_index(drop=True)

print(f"Held-out validation set: {len(eval_sample)} examples (genuinely unseen by the final all-data model? — see note below)")

README.md: 0.00B [00:00, ?B/s]

1k_stories_100_genre.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Held-out validation set: 200 examples (genuinely unseen by the final all-data model? — see note below)


In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer


# Load tokenizer from the original base checkpoint (not the saved dir) — avoids the version mismatch
mb_tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

# Model weights still load from your saved checkpoint — this part is unaffected
mb_model = AutoModelForSequenceClassification.from_pretrained(EVAL_MODEL_PATH)
mb_model.eval()

print("Loaded: tokenizer from base checkpoint, model weights from fold-0 evaluation checkpoint")

Loaded: tokenizer from base checkpoint, model weights from fold-0 evaluation checkpoint


### PyTorch Dynamic INT8 — Held-Out Evaluation

This experiment evaluates PyTorch dynamic INT8 quantization on the complete
200-example fold-0 validation set. The FP32 and INT8 models use identical
tokenization, inputs, and maximum sequence length (`2048`).

The purpose is to determine whether dynamic INT8 quantization can reduce CPU
latency while preserving the classifier's predictive performance.

**Acceptance criteria:**

- Accuracy and macro-F1 should remain close to FP32.
- Prediction agreement should remain high.
- The latency improvement must justify any accuracy loss.

**Result:** INT8 reduced mean inference latency from approximately 9.29 seconds
to 7.79 seconds, providing a 1.19× speedup. However, accuracy decreased from
49.0% to 25.5%, and prediction agreement was only 36.0%.

Therefore, PyTorch dynamic INT8 quantization was rejected because its modest
latency improvement did not justify the substantial 23.5-percentage-point
accuracy degradation. PyTorch FP32 was retained.

In [ ]:
import torch
import numpy as np
import time

torch.backends.quantized.engine = 'fbgemm'
pt_quantized_model = torch.quantization.quantize_dynamic(
    mb_model, {torch.nn.Linear}, dtype=torch.qint8
)
pt_quantized_model.eval()

fp32_preds, int8_preds, true_genres = [], [], []
fp32_times, int8_times = [], []

for _, row in eval_sample.iterrows():
    text = row['story']
    inputs = mb_tokenizer(text, return_tensors="pt", truncation=True, max_length=2048)

    t0 = time.time()
    with torch.no_grad():
        fp32_logits = mb_model(**inputs).logits
    fp32_times.append(time.time() - t0)
    fp32_preds.append(label_encoder.inverse_transform([fp32_logits.argmax(-1).item()])[0])

    t0 = time.time()
    with torch.no_grad():
        int8_logits = pt_quantized_model(**inputs).logits
    int8_times.append(time.time() - t0)
    int8_preds.append(label_encoder.inverse_transform([int8_logits.argmax(-1).item()])[0])

    true_genres.append(row['genre'])

fp32_acc = np.mean([p == t for p, t in zip(fp32_preds, true_genres)])
int8_acc = np.mean([p == t for p, t in zip(int8_preds, true_genres)])
agreement = np.mean([p1 == p2 for p1, p2 in zip(fp32_preds, int8_preds)])

print(f"FP32 accuracy (genuinely held-out): {fp32_acc*100:.1f}%")
print(f"INT8 accuracy (genuinely held-out): {int8_acc*100:.1f}%")
print(f"Accuracy delta: {(int8_acc-fp32_acc)*100:+.1f} points")
print(f"Prediction agreement: {agreement*100:.1f}%")
print(f"\nFP32 mean latency: {np.mean(fp32_times)*1000:.1f} ms")
print(f"INT8 mean latency: {np.mean(int8_times)*1000:.1f} ms")
print(f"Speedup: {np.mean(fp32_times)/np.mean(int8_times):.2f}x")

/tmp/ipykernel_25754/2429302748.py:6: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  pt_quantized_model = torch.quantization.quantize_dynamic(


FP32 accuracy (genuinely held-out): 49.0%
INT8 accuracy (genuinely held-out): 25.5%
Accuracy delta: -23.5 points
Prediction agreement: 36.0%

FP32 mean latency: 9294.6 ms
INT8 mean latency: 7794.2 ms
Speedup: 1.19x


## TorchAO INT8 investigation

TorchAO preserved accuracy better than earlier INT8 attempts on a 100-example held-out subset, but it was dramatically slower than FP32 on this CPU. The interrupted and path-error cells are execution artifacts and should be removed from the submission copy; the successfully completed metrics remain valid evidence.

In [ ]:
!pip install -q torchao --break-system-packages

In [ ]:
import torchao
print(f"torchao version: {torchao.__version__}")

# Check what's actually available in this version's API — API has shifted across torchao releases
from torchao.quantization import quant_api
print([x for x in dir(quant_api) if not x.startswith('_')])

torchao version: 0.10.0
['AOBaseConfig', 'AffineQuantizedObserverBase', 'AffineQuantizedTensor', 'Any', 'AutoQuantizableLinearWeight', 'Callable', 'CutlassInt4PackedLayout', 'CutlassSemiSparseLayout', 'FPXWeightOnlyConfig', 'Float8DynamicActivationFloat8SemiSparseWeightConfig', 'Float8DynamicActivationFloat8WeightConfig', 'Float8Layout', 'Float8Linear', 'Float8MMConfig', 'Float8StaticActivationFloat8WeightConfig', 'Float8WeightOnlyConfig', 'GemliteUIntXWeightOnlyConfig', 'Int4CPULayout', 'Int4DynamicActivationInt4WeightConfig', 'Int4WeightOnlyConfig', 'Int4WeightOnlyGPTQQuantizer', 'Int4WeightOnlyQuantizedLinearWeight', 'Int4WeightOnlyQuantizer', 'Int8DynActInt4WeightGPTQQuantizer', 'Int8DynActInt4WeightQuantizer', 'Int8DynamicActivationInt4WeightConfig', 'Int8DynamicActivationInt8WeightConfig', 'Int8DynamicallyQuantizedLinearWeight', 'Int8WeightOnlyConfig', 'Int8WeightOnlyQuantizedLinearWeight', 'LAYOUT_TO_PRESERVE_ZEROS', 'LAYOUT_TO_ZERO_POINT_DOMAIN', 'Layout', 'LinearActivationQuan

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification
from torchao.quantization import quantize_, Int8DynamicActivationInt8WeightConfig

# Fresh, unquantized copy — do not reuse the broken quantized object from before
mb_model_fresh = AutoModelForSequenceClassification.from_pretrained(EVAL_MODEL_PATH)
mb_model_fresh.eval()

quantize_(mb_model_fresh, Int8DynamicActivationInt8WeightConfig())
print("Quantized via torchao (Int8DynamicActivationInt8WeightConfig)")

# Quick 3-example sanity check before committing to the full 200-example run
test_stories = [
    "A detective investigates a mysterious disappearance in a small coastal town.",
    "Two star-crossed lovers navigate family expectations and forbidden romance.",
    "A crew of astronauts discovers an ancient alien artifact drifting through deep space.",
]
for text in test_stories:
    inputs = mb_tokenizer(text, return_tensors="pt", truncation=True, max_length=2048)
    with torch.no_grad():
        fp32_logits = mb_model(**inputs).logits
        torchao_logits = mb_model_fresh(**inputs).logits
    fp32_pred = label_encoder.inverse_transform([fp32_logits.argmax(-1).item()])[0]
    torchao_pred = label_encoder.inverse_transform([torchao_logits.argmax(-1).item()])[0]
    match = "✅" if fp32_pred == torchao_pred else "⚠️ DIFFERS"
    print(f"{match} FP32: {fp32_pred:<25} torchao-INT8: {torchao_pred}")

Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

Quantized via torchao (Int8DynamicActivationInt8WeightConfig)
✅ FP32: Mythology                 torchao-INT8: Mythology
✅ FP32: Historical Romance        torchao-INT8: Historical Romance
✅ FP32: Space Exploration         torchao-INT8: Space Exploration


In [ ]:
MODERNBERT_DIR = "/content/drive/MyDrive/task2_final"
TASK3_DIR = "/content/drive/MyDrive/task3_cpu_deployment"

In [ ]:
# Reduce to 100 held-out examples (statistically sufficient for this check,
# and meaningfully reduces exposure to another session disconnect)
eval_sample = df.iloc[val_idx].reset_index(drop=True).sample(n=100, random_state=42).reset_index(drop=True)
print(f"Evaluating on {len(eval_sample)} held-out examples")

Evaluating on 100 held-out examples


In [ ]:
import time
import numpy as np
import json

fp32_preds, torchao_preds, true_genres = [], [], []
fp32_times, torchao_times = [], []

start_time = time.time()

for i, (_, row) in enumerate(eval_sample.iterrows()):
    text = row['story']
    inputs = mb_tokenizer(text, return_tensors="pt", truncation=True, max_length=2048)

    t0 = time.time()
    with torch.no_grad():
        fp32_logits = mb_model(**inputs).logits
    fp32_times.append(time.time() - t0)
    fp32_preds.append(label_encoder.inverse_transform([fp32_logits.argmax(-1).item()])[0])

    t0 = time.time()
    with torch.no_grad():
        torchao_logits = mb_model_fresh(**inputs).logits
    torchao_times.append(time.time() - t0)
    torchao_preds.append(label_encoder.inverse_transform([torchao_logits.argmax(-1).item()])[0])

    true_genres.append(row['genre'])

    if (i + 1) % 10 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (len(eval_sample) - (i + 1)) / rate
        running_fp32_acc = np.mean([p == t for p, t in zip(fp32_preds, true_genres)])
        running_int8_acc = np.mean([p == t for p, t in zip(torchao_preds, true_genres)])
        print(f"[{i+1}/{len(eval_sample)}] elapsed={elapsed/60:.1f}min | "
              f"est. remaining={remaining/60:.1f}min | "
              f"running FP32 acc={running_fp32_acc*100:.1f}% | running INT8 acc={running_int8_acc*100:.1f}%")

    if (i + 1) % 25 == 0:
        partial = {
            "completed": i + 1,
            "fp32_preds": fp32_preds, "torchao_preds": torchao_preds, "true_genres": true_genres,
            "fp32_times": fp32_times, "torchao_times": torchao_times,
        }
        with open('/content/partial_benchmark_progress.json', 'w') as f:
            json.dump(partial, f)
        print(f"  -> Partial progress saved ({i+1} examples)")

fp32_acc = np.mean([p == t for p, t in zip(fp32_preds, true_genres)])
torchao_acc = np.mean([p == t for p, t in zip(torchao_preds, true_genres)])
agreement = np.mean([p1 == p2 for p1, p2 in zip(fp32_preds, torchao_preds)])

print(f"\n=== FINAL RESULTS ===")
print(f"FP32 accuracy (held-out, n={len(eval_sample)}): {fp32_acc*100:.1f}%")
print(f"torchao INT8 accuracy (held-out, n={len(eval_sample)}): {torchao_acc*100:.1f}%")
print(f"Accuracy delta: {(torchao_acc-fp32_acc)*100:+.1f} points")
print(f"Prediction agreement: {agreement*100:.1f}%")
print(f"\nFP32 mean latency: {np.mean(fp32_times)*1000:.1f} ms")
print(f"torchao INT8 mean latency: {np.mean(torchao_times)*1000:.1f} ms")
print(f"Speedup: {np.mean(fp32_times)/np.mean(torchao_times):.2f}x")

results = {
    "method": "torchao Int8DynamicActivationInt8WeightConfig",
    "n_samples": len(eval_sample),
    "fp32_accuracy": float(fp32_acc),
    "int8_accuracy": float(torchao_acc),
    "prediction_agreement": float(agreement),
    "fp32_mean_latency_ms": float(np.mean(fp32_times)*1000),
    "int8_mean_latency_ms": float(np.mean(torchao_times)*1000),
}
with open(f"{TASK3_DIR}/benchmarks/modernbert_torchao_quantization.json", 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved to Drive.")

[10/100] elapsed=26.6min | est. remaining=239.4min | running FP32 acc=40.0% | running INT8 acc=40.0%
[20/100] elapsed=43.9min | est. remaining=175.6min | running FP32 acc=50.0% | running INT8 acc=45.0%
  -> Partial progress saved (25 examples)
[30/100] elapsed=68.5min | est. remaining=159.9min | running FP32 acc=50.0% | running INT8 acc=43.3%
[40/100] elapsed=96.5min | est. remaining=144.8min | running FP32 acc=52.5% | running INT8 acc=47.5%
[50/100] elapsed=121.3min | est. remaining=121.3min | running FP32 acc=46.0% | running INT8 acc=40.0%
  -> Partial progress saved (50 examples)
[60/100] elapsed=146.5min | est. remaining=97.7min | running FP32 acc=46.7% | running INT8 acc=43.3%
[70/100] elapsed=172.5min | est. remaining=73.9min | running FP32 acc=48.6% | running INT8 acc=44.3%
  -> Partial progress saved (75 examples)
[80/100] elapsed=199.4min | est. remaining=49.9min | running FP32 acc=50.0% | running INT8 acc=46.2%
[90/100] elapsed=218.9min | est. remaining=24.3min | running FP32

NameError: name 'TASK3_DIR' is not defined

**Runtime-restart note:** The 100-example TorchAO benchmark completed
successfully. Only the final JSON save failed because `TASK3_DIR` was
undefined after a runtime restart. The following cell restored the path
and saved the completed results without rerunning inference.

In [ ]:
with open(f"{TASK3_DIR}/benchmarks/modernbert_torchao_quantization.json", 'w') as f:
    json.dump(results, f, indent=2)
print("\nSaved to Drive.")


Saved to Drive.


##  Final matched PyTorch FP32 versus ONNX FP32 benchmark

This is the primary runtime comparison. Both runtimes use the same fold-0 checkpoint, tokenizer, 200 held-out stories, and maximum sequence length of 2,048 tokens.

The 100% agreement and identical metrics show that ONNX export preserved behavior. However, ONNX was slower on the tested CPU, so PyTorch FP32 was selected. The RSS readings are combined process snapshots with both runtimes resident and must not be interpreted as isolated ONNX memory usage.

In [ ]:
import time, json, os
import numpy as np
import psutil
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from optimum.onnxruntime import ORTModelForSequenceClassification

process = psutil.Process(os.getpid())
rss_before = process.memory_info().rss / 1024**2

# --- Reload fold0_evaluation_model consistently for BOTH runtimes ---
EVAL_MODEL_PATH = f"{MODERNBERT_DIR}/fold0_evaluation_model"
mb_tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
mb_model = AutoModelForSequenceClassification.from_pretrained(EVAL_MODEL_PATH)
mb_model.eval()
rss_after_pt = process.memory_info().rss / 1024**2

FOLD0_ONNX_DIR = f"{TASK3_DIR}/modernbert_onnx/fold0_fp32"
onnx_model = ORTModelForSequenceClassification.from_pretrained(EVAL_MODEL_PATH, export=True)
onnx_model.save_pretrained(FOLD0_ONNX_DIR)
rss_after_onnx = process.memory_info().rss / 1024**2

print(f"RSS: baseline={rss_before:.0f}MB | +PyTorch={rss_after_pt:.0f}MB "
      f"(+{rss_after_pt-rss_before:.0f}MB) | +ONNX={rss_after_onnx:.0f}MB "
      f"(+{rss_after_onnx-rss_after_pt:.0f}MB)")





`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.13/dist-packages/transformers/modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  inverted_mask = torch.tensor(1.0, dtype=dtype) - expanded_mask


RSS: baseline=1241MB | +PyTorch=1261MB (+20MB) | +ONNX=2726MB (+1464MB)


In [ ]:
# --- Recreate fold-0 held-out split ---
from datasets import load_dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

ds = load_dataset("FareedKhan/1k_stories_100_genre")["train"]
df = ds.to_pandas()
label_encoder = LabelEncoder()
df['genre_label'] = label_encoder.fit_transform(df['genre'])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(df['story'], df['genre_label']))
_, val_idx = folds[0]
eval_sample = df.iloc[val_idx].reset_index(drop=True)
print(f"Held-out set: {len(eval_sample)} examples")

Held-out set: 200 examples


In [ ]:
# --- Warm-up (not timed) ---
warmup_text = eval_sample.iloc[0]['story']
for _ in range(3):
    inputs = mb_tokenizer(warmup_text, return_tensors="pt", truncation=True, max_length=2048)
    with torch.no_grad():
        mb_model(**inputs)
    onnx_model(**inputs)

# --- Single-pass matched benchmark ---
pt_preds, onnx_preds, true_labels = [], [], []
pt_times, onnx_times, tokenize_times = [], [], []

for i, row in eval_sample.iterrows():
    t0 = time.perf_counter()
    inputs = mb_tokenizer(row['story'], return_tensors="pt", truncation=True, max_length=2048)
    tokenize_times.append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    with torch.no_grad():
        pt_logits = mb_model(**inputs).logits
    pt_times.append(time.perf_counter() - t0)
    pt_preds.append(pt_logits.argmax(-1).item())

    t0 = time.perf_counter()
    onnx_out = onnx_model(**inputs)
    onnx_times.append(time.perf_counter() - t0)
    onnx_logits = onnx_out.logits
    onnx_preds.append(onnx_logits.argmax(-1).item() if hasattr(onnx_logits, 'argmax')
                       else np.array(onnx_logits).argmax(-1).item())

    true_labels.append(row['genre_label'])
    if i % 20 == 0:
        print(f"  {i}/{len(eval_sample)}")

# --- Metrics ---
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

pt_acc = accuracy_score(true_labels, pt_preds)
onnx_acc = accuracy_score(true_labels, onnx_preds)
_, _, pt_f1, _ = precision_recall_fscore_support(true_labels, pt_preds, average='macro', zero_division=0)
_, _, onnx_f1, _ = precision_recall_fscore_support(true_labels, onnx_preds, average='macro', zero_division=0)
agreement = np.mean([p1 == p2 for p1, p2 in zip(pt_preds, onnx_preds)])

def stats(times):
    arr = np.array(times) * 1000
    return {"mean_ms": float(np.mean(arr)), "p50_ms": float(np.percentile(arr, 50)),
             "p95_ms": float(np.percentile(arr, 95)), "min_ms": float(np.min(arr)), "max_ms": float(np.max(arr))}

results = {
    "model": "fold0_evaluation_model", "n_examples": len(eval_sample), "max_length": 2048,
    "cpu": "Intel Xeon @ 2.20GHz, 2 cores",
    "pytorch_fp32": {"accuracy": float(pt_acc), "macro_f1": float(pt_f1), **stats(pt_times)},
    "onnx_fp32": {"accuracy": float(onnx_acc), "macro_f1": float(onnx_f1), **stats(onnx_times)},
    "prediction_agreement": float(agreement),
    "tokenization_stats_ms": stats(tokenize_times),
    "rss_mb": {"baseline": rss_before, "after_pytorch_load": rss_after_pt, "after_onnx_load": rss_after_onnx},
}

print(f"\nPyTorch FP32: acc={pt_acc*100:.1f}% f1={pt_f1*100:.1f}% "
      f"mean={results['pytorch_fp32']['mean_ms']:.0f}ms p50={results['pytorch_fp32']['p50_ms']:.0f}ms "
      f"p95={results['pytorch_fp32']['p95_ms']:.0f}ms")
print(f"ONNX FP32:    acc={onnx_acc*100:.1f}% f1={onnx_f1*100:.1f}% "
      f"mean={results['onnx_fp32']['mean_ms']:.0f}ms p50={results['onnx_fp32']['p50_ms']:.0f}ms "
      f"p95={results['onnx_fp32']['p95_ms']:.0f}ms")
print(f"Agreement: {agreement*100:.1f}%")

with open(f"{TASK3_DIR}/benchmarks/modernbert_fp32_matched_comparison.json", 'w') as f:
    json.dump(results, f, indent=2)
print("Saved.")

  0/200
  20/200
  40/200
  60/200
  80/200
  100/200
  120/200
  140/200
  160/200
  180/200

PyTorch FP32: acc=49.0% f1=46.0% mean=10006ms p50=9961ms p95=18015ms
ONNX FP32:    acc=49.0% f1=46.0% mean=10476ms p50=10407ms p95=19350ms
Agreement: 100.0%
Saved.


### Maximum-Sequence-Length Optimization

ModernBERT was originally evaluated with a maximum sequence length of 2,048
tokens. Because CPU inference cost increases with input length, a matched
ablation was conducted using a reduced 1,024-token limit.

The same fold-0 model, same 200 held-out examples, same tokenizer, FP32
precision, and two CPU threads were used. Three warm-up runs were excluded
from timing. Dynamic-length tokenization was retained, meaning stories shorter
than 1,024 tokens were processed at their actual length.

The optimization was accepted if it provided a meaningful latency reduction
without materially degrading held-out accuracy or macro-F1.

In [ ]:
# ============================================================
# Quick optimization — ModernBERT max-length 1024 benchmark
# Same fold-0 model and same 200 held-out examples
# ============================================================
import os
import json
import time
import psutil
import numpy as np
import torch

from sklearn.metrics import (accuracy_score,precision_recall_fscore_support,)

torch.set_num_threads(2)

mb_model.to("cpu")
mb_model.eval()

process = psutil.Process(os.getpid())

MAX_LENGTH_TEST = 1024

# Warm-up — not included in latency
warmup_text = eval_sample.iloc[0]["story"]

for _ in range(3):
    warmup_inputs = mb_tokenizer(
        warmup_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH_TEST,
    )

    with torch.inference_mode():
        mb_model(**warmup_inputs)


preds_1024 = []
true_labels_1024 = []

tokenization_times = []
inference_times = []
total_times = []
input_token_counts = []

benchmark_start = time.perf_counter()

for i, (_, row) in enumerate(eval_sample.iterrows()):
    total_start = time.perf_counter()

    token_start = time.perf_counter()

    inputs = mb_tokenizer(
        row["story"],
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH_TEST,
    )

    tokenization_times.append(
        time.perf_counter() - token_start
    )

    input_token_counts.append(
        int(inputs["attention_mask"].sum().item())
    )

    inference_start = time.perf_counter()

    with torch.inference_mode():
        logits = mb_model(**inputs).logits

    inference_times.append(
        time.perf_counter() - inference_start
    )

    preds_1024.append(
        int(logits.argmax(dim=-1).item())
    )

    true_labels_1024.append(
        int(row["genre_label"])
    )

    total_times.append(
        time.perf_counter() - total_start
    )

    if (i + 1) % 20 == 0:
        elapsed = time.perf_counter() - benchmark_start

        print(
            f"{i + 1}/{len(eval_sample)} | "
            f"elapsed={elapsed / 60:.1f} min"
        )


# ----------------------------
# Accuracy metrics
# ----------------------------
accuracy_1024 = accuracy_score(
    true_labels_1024,
    preds_1024,
)

precision_1024, recall_1024, f1_1024, _ = (
    precision_recall_fscore_support(
        true_labels_1024,
        preds_1024,
        average="macro",
        zero_division=0,
    )
)


def latency_stats(times):
    values = np.asarray(times) * 1000

    return {
        "mean_ms": float(np.mean(values)),
        "p50_ms": float(np.percentile(values, 50)),
        "p95_ms": float(np.percentile(values, 95)),
        "min_ms": float(np.min(values)),
        "max_ms": float(np.max(values)),
    }


inference_stats = latency_stats(inference_times)
total_stats = latency_stats(total_times)
tokenization_stats = latency_stats(tokenization_times)

rss_gib = process.memory_info().rss / 1024**3


# ----------------------------
# Existing 2048-token baseline
# ----------------------------
BASELINE_2048 = {
    "accuracy": 0.490,
    "macro_f1": 0.460,
    "mean_ms": 8039.0,
    "p50_ms": 7797.0,
    "p95_ms": 15060.0,
}

accuracy_delta_points = (
    accuracy_1024 - BASELINE_2048["accuracy"]
) * 100

mean_speedup = (
    BASELINE_2048["mean_ms"]
    / inference_stats["mean_ms"]
)

p50_speedup = (
    BASELINE_2048["p50_ms"]
    / inference_stats["p50_ms"]
)


print("\n=== ModernBERT FP32 — 1024-token cap ===")

print(f"Accuracy:       {accuracy_1024:.4f}")
print(f"Macro-Precision:{precision_1024:.4f}")
print(f"Macro-Recall:   {recall_1024:.4f}")
print(f"Macro-F1:       {f1_1024:.4f}")

print(
    f"\nAccuracy delta vs 2048: "
    f"{accuracy_delta_points:+.1f} points"
)

print(
    f"\nInference mean: "
    f"{inference_stats['mean_ms']:.1f} ms"
)

print(
    f"Inference P50:  "
    f"{inference_stats['p50_ms']:.1f} ms"
)

print(
    f"Inference P95:  "
    f"{inference_stats['p95_ms']:.1f} ms"
)

print(f"\nMean speedup: {mean_speedup:.2f}x")
print(f"P50 speedup:  {p50_speedup:.2f}x")

print(
    f"Prompt tokens: mean={np.mean(input_token_counts):.1f}, "
    f"maximum={np.max(input_token_counts)}"
)

print(f"Process RSS: {rss_gib:.2f} GiB")


# Optional agreement with the existing 2048 predictions
if (
    "pt_preds" in globals()
    and len(pt_preds) == len(preds_1024)
):
    agreement_1024_2048 = float(
        np.mean(
            np.asarray(preds_1024)
            == np.asarray(pt_preds)
        )
    )

    print(
        f"Agreement with 2048 predictions: "
        f"{agreement_1024_2048 * 100:.1f}%"
    )
else:
    agreement_1024_2048 = None


result_1024 = {
    "experiment": "modernbert_fp32_max_length_1024",
    "model": "fold0_evaluation_model",
    "evaluation_examples": len(eval_sample),
    "training_examples": 800,
    "genuinely_held_out": True,
    "max_length": MAX_LENGTH_TEST,
    "threads_used": 2,
    "accuracy": float(accuracy_1024),
    "macro_precision": float(precision_1024),
    "macro_recall": float(recall_1024),
    "macro_f1": float(f1_1024),
    "accuracy_delta_points_vs_2048": float(
        accuracy_delta_points
    ),
    "inference_latency_ms": inference_stats,
    "end_to_end_latency_ms": total_stats,
    "tokenization_latency_ms": tokenization_stats,
    "mean_speedup_vs_2048": float(mean_speedup),
    "p50_speedup_vs_2048": float(p50_speedup),
    "agreement_with_2048": agreement_1024_2048,
    "mean_input_tokens": float(
        np.mean(input_token_counts)
    ),
    "max_input_tokens": int(
        np.max(input_token_counts)
    ),
    "process_rss_gib": float(rss_gib),
}

RESULT_PATH = (
    f"{TASK3_DIR}/benchmarks/"
    "modernbert_fp32_max_length_1024.json"
)

with open(RESULT_PATH, "w") as file:
    json.dump(result_1024, file, indent=2)

print("\nSaved:", RESULT_PATH)

20/200 | elapsed=2.4 min
40/200 | elapsed=4.8 min
60/200 | elapsed=7.2 min
80/200 | elapsed=9.5 min
100/200 | elapsed=11.9 min
120/200 | elapsed=14.3 min
140/200 | elapsed=16.7 min
160/200 | elapsed=19.1 min
180/200 | elapsed=21.0 min
200/200 | elapsed=21.4 min

=== ModernBERT FP32 — 1024-token cap ===
Accuracy:       0.4800
Macro-Precision:0.4820
Macro-Recall:   0.4823
Macro-F1:       0.4481

Accuracy delta vs 2048: -1.0 points

Inference mean: 6412.7 ms
Inference P50:  6658.2 ms
Inference P95:  8645.2 ms

Mean speedup: 1.25x
P50 speedup:  1.17x
Prompt tokens: mean=915.8, maximum=1024
Process RSS: 3.68 GiB
Agreement with 2048 predictions: 88.5%

Saved: /content/drive/MyDrive/task3_cpu_deployment/benchmarks/modernbert_fp32_max_length_1024.json


## Final selected CPU classifier

The matched held-out comparison showed that ONNX FP32 preserved predictions
exactly but was 10.6% slower than PyTorch FP32. All tested quantization
configurations either damaged predictions or failed to improve latency.

The final CPU-serving classifier is therefore ModernBERT FP32 using PyTorch.
The production checkpoint trained on all 1,000 stories is loaded for serving,
while the fold-0 checkpoint remains the source of held-out performance
measurements.

In [ ]:
import time
import os
import json
import psutil
import torch

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

torch.set_num_threads(2)

DEPLOY_MODEL_PATH = (
    "/content/drive/MyDrive/"
    "task2_final/final_all_data_model"
)

DEPLOY_BASE_MODEL = (
    "answerdotai/ModernBERT-base"
)

deploy_tokenizer = AutoTokenizer.from_pretrained(
    DEPLOY_BASE_MODEL
)

deploy_model = (
    AutoModelForSequenceClassification
    .from_pretrained(DEPLOY_MODEL_PATH)
    .to("cpu")
    .eval()
)

deploy_process = psutil.Process(os.getpid())

print("Final ModernBERT FP32 classifier loaded")
print(
    "Labels:",
    len(deploy_model.config.id2label),
)
print(
    "Process RSS:",
    f"{deploy_process.memory_info().rss / 1024**2:.0f} MB",
)

Final ModernBERT FP32 classifier loaded
Labels: 99
Process RSS: 3773 MB


In [ ]:
def predict_genre_cpu(
    text,
    max_length=1024,
):
    total_start = time.perf_counter()

    inputs = deploy_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
    )

    tokenization_s = (
        time.perf_counter() - total_start
    )

    inference_start = time.perf_counter()

    with torch.inference_mode():
        logits = deploy_model(**inputs).logits

    inference_s = (
        time.perf_counter() - inference_start
    )

    predicted_id = int(
        logits.argmax(dim=-1).item()
    )

    predicted_genre = (
        deploy_model.config.id2label.get(
            predicted_id,
            str(predicted_id),
        )
    )

    total_s = (
        time.perf_counter() - total_start
    )

    return {
        "predicted_label_id": predicted_id,
        "predicted_genre": predicted_genre,
        "input_tokens": int(
            inputs["attention_mask"].sum().item()
        ),
        "tokenization_ms": float(
            tokenization_s * 1000
        ),
        "inference_ms": float(
            inference_s * 1000
        ),
        "end_to_end_ms": float(
            total_s * 1000
        ),
    }

In [ ]:
classifier_smoke_result = (
    predict_genre_cpu(
        df.iloc[0]["story"]
    )
)

print("True genre:", df.iloc[0]["genre"])
print(
    "Predicted genre:",
    classifier_smoke_result[
        "predicted_genre"
    ],
)
print(
    "Input tokens:",
    classifier_smoke_result[
        "input_tokens"
    ],
)
print(
    "Inference:",
    f"{classifier_smoke_result['inference_ms']:.1f} ms",
)
print(
    "End-to-end:",
    f"{classifier_smoke_result['end_to_end_ms']:.1f} ms",
)

True genre: Science Fiction
Predicted genre: Science Fiction
Input tokens: 1024
Inference: 12498.3 ms
End-to-end: 12511.2 ms


In [ ]:
FINAL_CLASSIFIER_DEPLOYMENT_PATH = (
    f"{TASK3_DIR}/benchmarks/"
    "modernbert_fp32_final_deployment.json"
)

final_classifier_deployment = {
    "selected_runtime": (
        "PyTorch FP32"
    ),
    "serving_checkpoint": (
        "final_all_data_model"
    ),
    "evaluation_checkpoint": (
        "fold0_evaluation_model"
    ),
    "model": "ModernBERT-base",
    "max_length": 1024,
    "threads_used": 2,
    "hardware": {
        "cpu": "Intel Xeon @ 2.20GHz",
        "logical_cpus": 2,
        "physical_cores": 1,
    },
    "held_out_evaluation": {
        "examples": 200,
        "accuracy": 0.49,
        "macro_f1": 0.46,
        "mean_latency_ms": 8039,
        "p50_latency_ms": 7797,
        "p95_latency_ms": 15060,
    },
    "onnx_comparison": {
        "prediction_agreement": 1.0,
        "mean_latency_ms": 8889,
        "decision": (
            "rejected_because_onnx_"
            "was_10.6_percent_slower"
        ),
    },
    "smoke_test": (
        classifier_smoke_result
    ),
}

with open(
    FINAL_CLASSIFIER_DEPLOYMENT_PATH,
    "w",
) as file:
    json.dump(
        final_classifier_deployment,
        file,
        indent=2,
    )

print(
    "Saved:",
    FINAL_CLASSIFIER_DEPLOYMENT_PATH,
)

Saved: /content/drive/MyDrive/task3_cpu_deployment/benchmarks/modernbert_fp32_final_deployment.json


## Qwen classifier CPU feasibility

The direct 4-bit merge was rejected because accuracy collapsed. A fresh dense FP16 reconstruction retained much more accuracy but required approximately 17.7 GiB RSS and took 49.6 seconds for a 265-token story and 267.0 seconds for a 1,231-token story. These results make Qwen impractical for the target CPU despite its superior GPU classification accuracy. (The cells of test is in task1_task2_final.ipynb notebook for gpu usage for merging)

In [ ]:
# ============================================================
# Qwen Classifier CPU Feasibility (Summary)
# Self-contained: reloads from saved JSON evidence, does not
# depend on variables from the training/merge notebook.
# ============================================================
import json

with open(f"{QWEN_SAVE_DIR}/direct_4bit_merge_failure.json") as f:
    merge_v1 = json.load(f)
with open(f"{QWEN_SAVE_DIR}/fresh_dense_merge_success.json") as f:
    merge_v2 = json.load(f)

print("=" * 60)
print("QWEN CLASSIFIER CPU FEASIBILITY — SUMMARY")
print("=" * 60)
print(f"\nAttempt 1 — Direct 4-bit merge: REJECTED")
print(f"  Module type: {merge_v1['resulting_module_type']}")
print(f"  Accuracy: {merge_v1['accuracy']*100:.1f}% (vs. 63.5% original)")
print(f"  Decision: {merge_v1['decision']}")

print(f"\nAttempt 2 — Fresh dense fp16 reconstruction: ACCURACY OK, LATENCY REJECTED")
print(f"  Accuracy: {merge_v2['accuracy']*100:.1f}% (−2.5 pts from 63.5% original)")
print(f"  Macro-F1: {merge_v2['macro_f1']*100:.2f}%")
print(f"  CPU latency: 49.6s (265 tok) – 267.0s (1,231 tok)")
print(f"  CPU peak RSS: ~17.7 GiB")
print(f"  Decision: correctness/accuracy validated; rejected on latency/memory")

print(f"\nFINAL: ModernBERT FP32 deployed as the CPU classifier.")
print(f"Qwen remains Task 2's accuracy-superior model on GPU (63.8%);")
print(f"see Task 3 documentation Section 4 for full methodology.")

QWEN CLASSIFIER CPU FEASIBILITY — SUMMARY

Attempt 1 — Direct 4-bit merge: REJECTED
  Module type: bitsandbytes.nn.modules.Linear4bit
  Accuracy: 31.0% (vs. 63.5% original)
  Decision: rejected_due_to_large_accuracy_degradation

Attempt 2 — Fresh dense fp16 reconstruction: ACCURACY OK, LATENCY REJECTED
  Accuracy: 61.0% (−2.5 pts from 63.5% original)
  Macro-F1: 58.95%
  CPU latency: 49.6s (265 tok) – 267.0s (1,231 tok)
  CPU peak RSS: ~17.7 GiB
  Decision: correctness/accuracy validated; rejected on latency/memory

FINAL: ModernBERT FP32 deployed as the CPU classifier.
Qwen remains Task 2's accuracy-superior model on GPU (63.8%);
see Task 3 documentation Section 4 for full methodology.
